# 01. Floating Point Anatomy, Subnormal Traps, and Stochastic Logic

- **Category:** 13_precision_and_numerics
- **Lineage & Citations:** Gaines (1969 *Stochastic Computing*); IEEE 754 Standard for Floating-Point Arithmetic (1985/2019); Kalamkar et al. (2019 *A Study of BFLOAT16 for Deep Learning Training*).
- **Purpose:** Formulate and verify IEEE 754 floating-point anatomy, BF16 round-to-nearest-even conversion, Flush-To-Zero (FTZ) / Denormals-Are-Zero (DAZ) subnormal handling, and multiplier-free Stochastic Computing bitstream logic (AND/XNOR gates).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [07. Profiling Roofline Analysis CUDA Graphs and Fused Optimizers](../12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. Posit Quire Takum and Logarithmic Number Systems LNS](02_posit_quire_takum_and_logarithmic_number_systems_lns.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not accumulate small gradient updates into large parameters in pure BF16 without Kahan compensation or stochastic rounding (causes gradient stagnation); do not leave subnormal microcode traps unhandled in real-time inference.

## Mathematical Foundations & Governing Equations

1. **IEEE 754 Real Representation**:
   $$x = (-1)^s \times 2^{E - B} \times (1 + M)$$
2. **Stochastic Computing Unipolar Multiplication** (AND gate):
   $$\hat{p} = \frac{1}{L} \sum_{k=1}^L (b_{1, k} \land b_{2, k}) \approx p_1 \cdot p_2$$
3. **Stochastic Computing Bipolar Multiplication** (XNOR gate):
   $$\hat{x} = 2 \left( \frac{1}{L} \sum_{k=1}^L [b_{1, k} b_{2, k} + (1 - b_{1, k})(1 - b_{2, k})] \right) - 1 \approx x_1 \cdot x_2$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def ftz_daz_simulate(x, threshold=1e-4):
    return torch.where(torch.abs(x) < float(threshold), torch.zeros_like(x), x)

def fp32_to_bf16_rtne(x):
    return x.bfloat16().float()

def stochastic_unipolar_mul(p1, p2, u1, u2):
    b1 = (u1 < p1.unsqueeze(1)).float()
    b2 = (u2 < p2.unsqueeze(1)).float()
    return torch.mean(b1 * b2, dim=-1)

def stochastic_bipolar_mul(x1, x2, u1, u2):
    p1 = (x1 + 1.0) * 0.5
    p2 = (x2 + 1.0) * 0.5
    b1 = (u1 < p1.unsqueeze(1)).float()
    b2 = (u2 < p2.unsqueeze(1)).float()
    b_xnor = b1 * b2 + (1.0 - b1) * (1.0 - b2)
    return 2.0 * torch.mean(b_xnor, dim=-1) - 1.0

In [ ]:
# Verify FTZ, BF16 conversion, and stochastic logic against golden reference tensors
ftz = ftz_daz_simulate(weights["raw_vals"], threshold=1e-4)
assert torch.max(torch.abs(ftz - weights["ftz_ref"])) < 1e-5

bf16 = fp32_to_bf16_rtne(weights["bf16_in"])
assert torch.max(torch.abs(bf16 - weights["bf16_ref"])) < 1e-5

unipolar = stochastic_unipolar_mul(weights["p1"], weights["p2"], weights["u1"], weights["u2"])
assert torch.max(torch.abs(unipolar - weights["unipolar_ref"])) < 1e-5

bipolar = stochastic_bipolar_mul(weights["x1"], weights["x2"], weights["u1"], weights["u2"])
assert torch.max(torch.abs(bipolar - weights["bipolar_ref"])) < 1e-5
print("All precision, FTZ, and stochastic computing parity assertions verified!")

In [ ]:
# Microbenchmark: Bitstream stochastic multiplication
p1, p2, u1, u2 = weights["p1"], weights["p2"], weights["u1"], weights["u2"]
t0 = time.perf_counter()
for _ in range(500):
    _ = stochastic_unipolar_mul(p1, p2, u1, u2)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Stochastic bitstream multiplication latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: BF16 Underflow & Stagnation Trap
base_weight = torch.tensor(16384.0, dtype=torch.bfloat16)
small_update = torch.tensor(1.0, dtype=torch.bfloat16)
stagnant_weight = base_weight + small_update
print(f"BF16 Addition: {base_weight.item()} + {small_update.item()} = {stagnant_weight.item()}")
assert stagnant_weight.item() == 16384.0, "BF16 addition must demonstrate numerical stagnation!"

# FTZ Demonstration: Subnormal numbers are flushed to zero
subnormals = torch.tensor([1e-5, -1e-5, 1e-6])
flushed = ftz_daz_simulate(subnormals, threshold=1e-4)
assert torch.all(flushed == 0.0), "Subnormals must flush to zero!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **BF16 vs FP16**: BF16 preserves FP32 dynamic range ($10^{\pm 38}$) preventing gradient overflow, but exhibits higher machine epsilon ($\epsilon \approx 7.8 \times 10^{-3}$), risking update stagnation.
2. **FTZ / DAZ**: Flushing subnormal numbers to zero prevents catastrophic hardware microcode trap stalls ($100\times$ latency spike).
3. **Stochastic Computing**: Transforms multi-bit arithmetic multipliers into single-gate AND / XNOR operations over Bernoulli bitstreams.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.md](../../markdown/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.md)
- **PyTorch 2.x Module:** [01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.py](../../pytorch2/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.py)
- **Functional JAX Module:** [01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.py](../../jax/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.py)
- **Native Rust Candle Track:** [01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.rs](../../candle/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.rs)
- **Hardware Kernel (CPU / WGSL):** [01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.rs](../../kernels/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.rs)
- **Golden SafeTensors:** [01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.safetensors](../../weights/13_precision_and_numerics/01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Profiling Roofline Analysis CUDA Graphs and Fused Optimizers](../12_compilation_and_performance/07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.ipynb) | [Master Index](../../README.md#13_precision_and_numerics) | [Next: 02. Posit Quire Takum and Logarithmic Number Systems LNS](02_posit_quire_takum_and_logarithmic_number_systems_lns.ipynb) &rarr;